# 🌐 opendata.paris API Integration — Step 2
## Real Estate Analysis in Paris

**Author:** Ahmed Maala  
**Team:** Ahmed Maala & Natalja Voth  
**Date:** May 12, 2026  
**Project:** Liora Data Engineering Training

---

## 🎯 Objectives

In this notebook, we will integrate with the **opendata.paris** API to 
fetch real estate-related datasets that complement our DVF analysis.

### Datasets to fetch:

1. **Risk Sectors** — Flood risk areas in Paris
2. **Rent Control** — Legal rent prices by district
3. **Open Spaces** — Spaces to be planted (green spaces)

### Why use the API?

- ✅ **Automated**: No manual downloads
- ✅ **Up-to-date**: Always latest data
- ✅ **Flexible**: Can filter, sort, and paginate
- ✅ **Professional**: Industry-standard approach

---

## 📚 API Documentation

**Base URL:** `https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/`

**Format:** REST API returning JSON

**Authentication:** None required (public API) ✅

In [1]:
# ===== Standard Libraries =====
import os
import sys
import json
from pathlib import Path

# ===== Data Manipulation =====
import pandas as pd
import numpy as np

# ===== HTTP Requests for API =====
import requests

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Display Settings =====
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

# ===== Plot Style =====
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

# ===== Confirmation =====
print("✅ All libraries imported successfully!")
print(f"📦 pandas version: {pd.__version__}")
print(f"📦 requests version: {requests.__version__}")
print(f"\n🎯 Ready to connect to opendata.paris API!")

✅ All libraries imported successfully!
📦 pandas version: 3.0.2
📦 requests version: 2.33.1

🎯 Ready to connect to opendata.paris API!


## 🔌 Step 1: Test API Connection

Before fetching the real datasets, let's test if the API is reachable 
and understand its response structure.

We will make a simple request to fetch metadata about one dataset.

In [2]:
# ===== Test API Connection =====
print("🔌 Testing API connection...\n")

# API endpoint for "logement-encadrement-des-loyers" dataset
BASE_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets"
TEST_DATASET = "logement-encadrement-des-loyers"
TEST_URL = f"{BASE_URL}/{TEST_DATASET}/records?limit=1"

print(f"📍 URL: {TEST_URL}\n")

# Make the request
try:
    response = requests.get(TEST_URL, timeout=30)
    
    # Check status
    print(f"📊 Status code: {response.status_code}")
    
    if response.status_code == 200:
        print("✅ Connection successful!\n")
        
        # Parse JSON
        data = response.json()
        
        # Show response structure
        print(f"📋 Response keys: {list(data.keys())}")
        
        if 'total_count' in data:
            print(f"📊 Total records available: {data['total_count']:,}")
        
        if 'results' in data and len(data['results']) > 0:
            print(f"\n🔍 First record (sample):")
            print(json.dumps(data['results'][0], indent=2, ensure_ascii=False)[:500])
            print("...")
    else:
        print(f"❌ Error: HTTP {response.status_code}")
        print(response.text[:500])
        
except requests.exceptions.RequestException as e:
    print(f"❌ Connection error: {e}")

🔌 Testing API connection...

📍 URL: https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/logement-encadrement-des-loyers/records?limit=1

📊 Status code: 200
✅ Connection successful!

📋 Response keys: ['total_count', 'results']
📊 Total records available: 17,920

🔍 First record (sample):
{
  "annee": "2025",
  "id_zone": 5,
  "id_quartier": 38,
  "nom_quartier": "Porte-Saint-Denis",
  "piece": 2,
  "epoque": "1946-1970",
  "meuble_txt": "non meublé",
  "ref": 27.4,
  "max": 32.9,
  "min": 19.2,
  "ville": "PARIS",
  "code_grand_quartier": 7511038,
  "geo_shape": {
    "type": "Feature",
    "geometry": {
      "coordinates": [
        [
          [
            2.3553447343852665,
            48.871262685264895
          ],
          [
            2.3542605804166112,
            
...


## 📊 Step 2: Fetch the 3 Real Estate Datasets

We will now fetch the 3 datasets needed for our project:

| # | Dataset | Purpose |
|---|---------|---------|
| 1 | **Rent Control** | Legal rent prices in Paris |
| 2 | **Risk Sectors** | Flood risk areas (PPRI) |
| 3 | **Open Spaces** | Green spaces to be planted |

### Strategy

- Fetch each dataset with pagination (in batches of 100 records)
- Convert JSON responses to pandas DataFrames
- Save raw data to `data/raw/`

In [3]:
# ===== Helper function to fetch ALL records from a dataset =====
def fetch_opendata_paris(dataset_id, max_records=None, batch_size=100):
    """
    Fetch all records from an opendata.paris dataset with pagination.
    
    Parameters:
    -----------
    dataset_id : str
        The dataset identifier (e.g., 'logement-encadrement-des-loyers')
    max_records : int or None
        Maximum records to fetch. None = fetch all.
    batch_size : int
        Records per API call (max 100 for opendata.paris)
    
    Returns:
    --------
    pandas.DataFrame with all records
    """
    base_url = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets"
    all_records = []
    offset = 0
    
    print(f"🔄 Fetching dataset: {dataset_id}")
    print(f"   Batch size: {batch_size}")
    
    # First call to get total count
    url = f"{base_url}/{dataset_id}/records?limit={batch_size}&offset={offset}"
    response = requests.get(url, timeout=30)
    
    if response.status_code != 200:
        print(f"❌ Error: HTTP {response.status_code}")
        return None
    
    data = response.json()
    total_count = data.get('total_count', 0)
    print(f"   📊 Total records available: {total_count:,}")
    
    # Determine how many to fetch
    if max_records is None:
        target = total_count
    else:
        target = min(max_records, total_count)
    
    print(f"   🎯 Will fetch: {target:,} records\n")
    
    # Add first batch
    all_records.extend(data.get('results', []))
    offset += batch_size
    
    # Fetch remaining batches
    while offset < target:
        url = f"{base_url}/{dataset_id}/records?limit={batch_size}&offset={offset}"
        response = requests.get(url, timeout=30)
        
        if response.status_code != 200:
            print(f"   ⚠️  Error at offset {offset}: HTTP {response.status_code}")
            break
        
        data = response.json()
        batch = data.get('results', [])
        
        if not batch:
            break
        
        all_records.extend(batch)
        offset += batch_size
        
        # Progress
        progress = min(offset, target)
        percent = (progress / target) * 100
        print(f"   📥 Progress: {progress:,} / {target:,} ({percent:.1f}%)")
    
    print(f"\n✅ Fetched {len(all_records):,} records successfully!")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_records)
    print(f"📋 DataFrame shape: {df.shape}")
    
    return df


print("✅ Function 'fetch_opendata_paris' defined!")
print("📝 Ready to fetch datasets...")

✅ Function 'fetch_opendata_paris' defined!
📝 Ready to fetch datasets...


### 📊 Dataset 1: Rent Control (Legal Rent Prices)

**Dataset ID:** `logement-encadrement-des-loyers`

**Description:** Maximum legal rent prices per district in Paris, 
used for the "Rent Control Law" (Encadrement des Loyers).

**Importance:** Essential for understanding the legal rental market in Paris.

In [4]:
# ===== Fetch Dataset 1: Rent Control =====
df_rent = fetch_opendata_paris(
    dataset_id="logement-encadrement-des-loyers",
    max_records=None,  # Fetch all
    batch_size=100
)

# Show preview
print("\n🔍 First 5 rows:")
df_rent.head()

🔄 Fetching dataset: logement-encadrement-des-loyers
   Batch size: 100
   📊 Total records available: 17,920
   🎯 Will fetch: 17,920 records

   📥 Progress: 200 / 17,920 (1.1%)
   📥 Progress: 300 / 17,920 (1.7%)
   📥 Progress: 400 / 17,920 (2.2%)
   📥 Progress: 500 / 17,920 (2.8%)
   📥 Progress: 600 / 17,920 (3.3%)
   📥 Progress: 700 / 17,920 (3.9%)
   📥 Progress: 800 / 17,920 (4.5%)
   📥 Progress: 900 / 17,920 (5.0%)
   📥 Progress: 1,000 / 17,920 (5.6%)
   📥 Progress: 1,100 / 17,920 (6.1%)
   📥 Progress: 1,200 / 17,920 (6.7%)
   📥 Progress: 1,300 / 17,920 (7.3%)
   📥 Progress: 1,400 / 17,920 (7.8%)
   📥 Progress: 1,500 / 17,920 (8.4%)
   📥 Progress: 1,600 / 17,920 (8.9%)
   📥 Progress: 1,700 / 17,920 (9.5%)
   📥 Progress: 1,800 / 17,920 (10.0%)
   📥 Progress: 1,900 / 17,920 (10.6%)
   📥 Progress: 2,000 / 17,920 (11.2%)
   📥 Progress: 2,100 / 17,920 (11.7%)
   📥 Progress: 2,200 / 17,920 (12.3%)
   📥 Progress: 2,300 / 17,920 (12.8%)
   📥 Progress: 2,400 / 17,920 (13.4%)
   📥 Progress: 2,

,annee,id_zone,id_quartier,nom_quartier,piece,epoque,meuble_txt,ref,max,min,ville,code_grand_quartier,geo_shape,geo_point_2d
0,2025,5,38,Porte-Saint-Denis,2,1946-1970,non meublé,27.4,32.9,19.2,PARIS,7511038,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.3553447343852665, 48.871262685264895], [2.3...","{'lon': 2.352282894951218, 'lat': 48.87361766097465}"
1,2025,11,77,Belleville,1,1971-1990,non meublé,27.8,33.4,19.5,PARIS,7512077,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.3832266916750324, 48.86709755024514], [2.38...","{'lon': 2.3875492398498035, 'lat': 48.87153120058614}"
2,2023,8,57,Saint-Lambert,1,1971-1990,meublé,28.3,34.0,19.8,PARIS,7511557,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.3042487966491323, 48.84040838273814], [2.30...","{'lon': 2.296919974448867, 'lat': 48.83429362836976}"
3,2024,10,67,Batignolles,1,1946-1970,non meublé,26.1,31.3,18.3,PARIS,7511767,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.317210277038245, 48.890259814260936], [2.31...","{'lon': 2.313856169006357, 'lat': 48.888481513920595}"
4,2024,9,69,Grandes-Carrières,4,1946-1970,meublé,21.1,25.3,14.8,PARIS,7511869,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.340216773881389, 48.89310518571166], [2.339...","{'lon': 2.334363089913081, 'lat': 48.89257777114592}"


### 📊 Dataset 2: Risk Sectors (Flood Risk Areas)

**Dataset ID:** `plu-secteurs-de-risques-delimites-par-le-ppri`

**Description:** Flood risk zones in Paris, delimited by the PPRI 
(Plan de Prévention des Risques d'Inondation).

**Importance:** Critical for understanding property risks and prices 
in areas near the Seine river.

In [7]:
# ===== Fetch Dataset 2: Risk Sectors (PLU Bioclimatique) =====
df_risk = fetch_opendata_paris(
    dataset_id="plub_pprizone",
    max_records=None,
    batch_size=100
)

# Show preview only if data was fetched successfully
if df_risk is not None:
    print("\n🔍 First 5 rows:")
    df_risk.head()
else:
    print("\n⚠️ No data fetched. Check dataset ID and try again.")

🔄 Fetching dataset: plub_pprizone
   Batch size: 100
   📊 Total records available: 1,653
   🎯 Will fetch: 1,653 records

   📥 Progress: 200 / 1,653 (12.1%)
   📥 Progress: 300 / 1,653 (18.1%)
   📥 Progress: 400 / 1,653 (24.2%)
   📥 Progress: 500 / 1,653 (30.2%)
   📥 Progress: 600 / 1,653 (36.3%)
   📥 Progress: 700 / 1,653 (42.3%)
   📥 Progress: 800 / 1,653 (48.4%)
   📥 Progress: 900 / 1,653 (54.4%)
   📥 Progress: 1,000 / 1,653 (60.5%)
   📥 Progress: 1,100 / 1,653 (66.5%)
   📥 Progress: 1,200 / 1,653 (72.6%)
   📥 Progress: 1,300 / 1,653 (78.6%)
   📥 Progress: 1,400 / 1,653 (84.7%)
   📥 Progress: 1,500 / 1,653 (90.7%)
   📥 Progress: 1,600 / 1,653 (96.8%)
   📥 Progress: 1,653 / 1,653 (100.0%)

✅ Fetched 1,653 records successfully!
📋 DataFrame shape: (1653, 7)

🔍 First 5 rows:


### 📊 Dataset 3: Open Spaces (Green Spaces to be Planted)

**Dataset ID:** `plub_elpv`

**Description:** Open spaces in Paris designated to be planted 
with vegetation under the PLU Bioclimatique 2024.

**Importance:** Green spaces significantly impact real estate prices 
and quality of life in urban environments.

In [8]:
# ===== Fetch Dataset 3: Open Spaces (PLU Bioclimatique) =====
df_green = fetch_opendata_paris(
    dataset_id="plub_elpv",
    max_records=None,
    batch_size=100
)

# Show preview only if data was fetched successfully
if df_green is not None:
    print("\n🔍 First 5 rows:")
    df_green.head()
else:
    print("\n⚠️ No data fetched. Check dataset ID and try again.")

🔄 Fetching dataset: plub_elpv
   Batch size: 100
   📊 Total records available: 7,268
   🎯 Will fetch: 7,268 records

   📥 Progress: 200 / 7,268 (2.8%)
   📥 Progress: 300 / 7,268 (4.1%)
   📥 Progress: 400 / 7,268 (5.5%)
   📥 Progress: 500 / 7,268 (6.9%)
   📥 Progress: 600 / 7,268 (8.3%)
   📥 Progress: 700 / 7,268 (9.6%)
   📥 Progress: 800 / 7,268 (11.0%)
   📥 Progress: 900 / 7,268 (12.4%)
   📥 Progress: 1,000 / 7,268 (13.8%)
   📥 Progress: 1,100 / 7,268 (15.1%)
   📥 Progress: 1,200 / 7,268 (16.5%)
   📥 Progress: 1,300 / 7,268 (17.9%)
   📥 Progress: 1,400 / 7,268 (19.3%)
   📥 Progress: 1,500 / 7,268 (20.6%)
   📥 Progress: 1,600 / 7,268 (22.0%)
   📥 Progress: 1,700 / 7,268 (23.4%)
   📥 Progress: 1,800 / 7,268 (24.8%)
   📥 Progress: 1,900 / 7,268 (26.1%)
   📥 Progress: 2,000 / 7,268 (27.5%)
   📥 Progress: 2,100 / 7,268 (28.9%)
   📥 Progress: 2,200 / 7,268 (30.3%)
   📥 Progress: 2,300 / 7,268 (31.6%)
   📥 Progress: 2,400 / 7,268 (33.0%)
   📥 Progress: 2,500 / 7,268 (34.4%)
   📥 Progress: 2,

## 📊 Step 3: Summary of All Datasets

Let's compare the 3 fetched datasets from opendata.paris:

In [9]:
# ===== Summary of all datasets =====
print("📊 Datasets Summary from opendata.paris")
print("=" * 70)

datasets = {
    "1. Rent Control (logement-encadrement-des-loyers)": df_rent,
    "2. Risk Sectors (plub_pprizone)": df_risk,
    "3. Open Spaces (plub_elpv)": df_green
}

total_records = 0
total_memory = 0

for name, df in datasets.items():
    if df is not None:
        memory_mb = df.memory_usage(deep=True).sum() / (1024**2)
        total_records += len(df)
        total_memory += memory_mb
        
        print(f"\n🔹 {name}")
        print(f"   📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"   💾 Memory: {memory_mb:.2f} MB")
        print(f"   📋 Columns: {list(df.columns)}")
    else:
        print(f"\n❌ {name}: Failed to fetch")

print("\n" + "=" * 70)
print(f"📈 TOTAL: {total_records:,} records, {total_memory:.2f} MB")
print(f"✅ All datasets fetched successfully!")

📊 Datasets Summary from opendata.paris

🔹 1. Rent Control (logement-encadrement-des-loyers)
   📊 Shape: 10,000 rows × 14 columns
   💾 Memory: 7.07 MB
   📋 Columns: ['annee', 'id_zone', 'id_quartier', 'nom_quartier', 'piece', 'epoque', 'meuble_txt', 'ref', 'max', 'min', 'ville', 'code_grand_quartier', 'geo_shape', 'geo_point_2d']

🔹 2. Risk Sectors (plub_pprizone)
   📊 Shape: 1,653 rows × 7 columns
   💾 Memory: 0.74 MB
   📋 Columns: ['objectid', 'zonage', 'n_sq_pprizone', 'st_area_shape', 'st_perimeter_shape', 'geo_shape', 'geo_point_2d']

🔹 3. Open Spaces (plub_elpv)
   📊 Shape: 7,268 rows × 10 columns
   💾 Memory: 3.75 MB
   📋 Columns: ['objectid', 'n_sq_elpv', 'n_sq_ca', 'c_sec', 'n_pc', 'c_asp', 'st_area_shape', 'st_perimeter_shape', 'geo_shape', 'geo_point_2d']

📈 TOTAL: 18,921 records, 11.56 MB
✅ All datasets fetched successfully!


## 💾 Step 4: Save Datasets to `data/raw/`

We save all 3 datasets in **two formats**:

- **CSV** — Easy to read and open in Excel
- **CSV.gz** — Compressed version (smaller file size)

This makes the data available for the next steps of the project 
(Database Modeling, ETL, Power BI Dashboard).

In [10]:
# ===== Save all datasets =====
PROJECT_ROOT = Path(r"C:\Users\ahmed\Documents\Projects\real-estate-paris")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("💾 Saving datasets to data/raw/...\n")

datasets_to_save = {
    "paris_rent_control": df_rent,
    "paris_risk_sectors": df_risk,
    "paris_open_spaces": df_green
}

for name, df in datasets_to_save.items():
    if df is not None:
        # Save as CSV
        csv_path = RAW_DIR / f"{name}.csv"
        df.to_csv(csv_path, index=False)
        
        # Save as CSV.gz (compressed)
        gz_path = RAW_DIR / f"{name}.csv.gz"
        df.to_csv(gz_path, index=False, compression='gzip')
        
        csv_mb = csv_path.stat().st_size / (1024**2)
        gz_mb = gz_path.stat().st_size / (1024**2)
        compression_ratio = (1 - gz_mb / csv_mb) * 100 if csv_mb > 0 else 0
        
        print(f"✅ {name}")
        print(f"   📄 CSV:  {csv_mb:.2f} MB → {csv_path}")
        print(f"   📦 GZ:   {gz_mb:.2f} MB ({compression_ratio:.1f}% smaller)")
        print()

print("=" * 70)
print(f"✅ All datasets saved successfully!")
print(f"📍 Location: {RAW_DIR}")

💾 Saving datasets to data/raw/...

✅ paris_rent_control
   📄 CSV:  43.24 MB → C:\Users\ahmed\Documents\Projects\real-estate-paris\data\raw\paris_rent_control.csv
   📦 GZ:   14.32 MB (66.9% smaller)

✅ paris_risk_sectors
   📄 CSV:  1.94 MB → C:\Users\ahmed\Documents\Projects\real-estate-paris\data\raw\paris_risk_sectors.csv
   📦 GZ:   0.68 MB (64.8% smaller)

✅ paris_open_spaces
   📄 CSV:  4.43 MB → C:\Users\ahmed\Documents\Projects\real-estate-paris\data\raw\paris_open_spaces.csv
   📦 GZ:   1.45 MB (67.2% smaller)

✅ All datasets saved successfully!
📍 Location: C:\Users\ahmed\Documents\Projects\real-estate-paris\data\raw


## 🔍 Step 5: Quick Data Exploration

Now that we have the 3 datasets, let's explore each one quickly:

### Questions to answer:
1. **What does the data look like?** (columns, types, samples)
2. **Are there missing values?**
3. **What's the geographic distribution?**
4. **What insights can we extract?**

### 🏠 Dataset 1: Rent Control Analysis

Let's understand the rent prices in Paris.

In [11]:
# ===== Explore df_rent (Rent Control) =====
print("=" * 70)
print("🏠 RENT CONTROL DATASET")
print("=" * 70)

print(f"\n📊 Shape: {df_rent.shape[0]:,} rows × {df_rent.shape[1]} columns")
print(f"\n📋 Columns ({len(df_rent.columns)}):")
for i, col in enumerate(df_rent.columns, 1):
    print(f"   {i:2d}. {col} ({df_rent[col].dtype})")

print(f"\n❓ Missing values:")
missing = df_rent.isnull().sum()
missing_pct = (missing / len(df_rent)) * 100
for col in df_rent.columns:
    if missing[col] > 0:
        print(f"   • {col}: {missing[col]:,} ({missing_pct[col]:.1f}%)")
    else:
        print(f"   ✅ {col}: no missing")

print(f"\n📅 Year range:")
if 'annee' in df_rent.columns:
    years = df_rent['annee'].value_counts().sort_index()
    for year, count in years.items():
        print(f"   • {year}: {count:,} records")

🏠 RENT CONTROL DATASET

📊 Shape: 10,000 rows × 14 columns

📋 Columns (14):
    1. annee (str)
    2. id_zone (int64)
    3. id_quartier (int64)
    4. nom_quartier (str)
    5. piece (int64)
    6. epoque (str)
    7. meuble_txt (str)
    8. ref (float64)
    9. max (float64)
   10. min (float64)
   11. ville (str)
   12. code_grand_quartier (int64)
   13. geo_shape (object)
   14. geo_point_2d (object)

❓ Missing values:
   ✅ annee: no missing
   ✅ id_zone: no missing
   ✅ id_quartier: no missing
   ✅ nom_quartier: no missing
   ✅ piece: no missing
   ✅ epoque: no missing
   ✅ meuble_txt: no missing
   ✅ ref: no missing
   ✅ max: no missing
   ✅ min: no missing
   ✅ ville: no missing
   ✅ code_grand_quartier: no missing
   ✅ geo_shape: no missing
   ✅ geo_point_2d: no missing

📅 Year range:
   • 2019: 1,427 records
   • 2020: 1,437 records
   • 2021: 1,401 records
   • 2022: 1,471 records
   • 2023: 1,430 records
   • 2024: 1,422 records
   • 2025: 1,412 records


### 💰 Rent Prices Analysis

In [12]:
# ===== Rent prices statistics =====
print("💰 RENT PRICE STATISTICS (€/m²)")
print("=" * 70)

# Make sure we have the right columns
price_cols = ['ref', 'max', 'min']
for col in price_cols:
    if col in df_rent.columns:
        # Convert to numeric if needed
        df_rent[col] = pd.to_numeric(df_rent[col], errors='coerce')

print(f"\n📊 Reference Price (€/m²):")
print(f"   • Min:  €{df_rent['ref'].min():.2f}")
print(f"   • Max:  €{df_rent['ref'].max():.2f}")
print(f"   • Mean: €{df_rent['ref'].mean():.2f}")
print(f"   • Median: €{df_rent['ref'].median():.2f}")

print(f"\n🏘️ Top 10 most expensive neighborhoods (by avg reference price):")
if 'nom_quartier' in df_rent.columns:
    top_expensive = df_rent.groupby('nom_quartier')['ref'].mean().sort_values(ascending=False).head(10)
    for i, (quartier, price) in enumerate(top_expensive.items(), 1):
        print(f"   {i:2d}. {quartier}: €{price:.2f}/m²")

print(f"\n🏘️ Top 10 cheapest neighborhoods:")
if 'nom_quartier' in df_rent.columns:
    top_cheap = df_rent.groupby('nom_quartier')['ref'].mean().sort_values().head(10)
    for i, (quartier, price) in enumerate(top_cheap.items(), 1):
        print(f"   {i:2d}. {quartier}: €{price:.2f}/m²")

💰 RENT PRICE STATISTICS (€/m²)

📊 Reference Price (€/m²):
   • Min:  €14.30
   • Max:  €41.80
   • Mean: €26.40
   • Median: €26.10

🏘️ Top 10 most expensive neighborhoods (by avg reference price):
    1. Saint-Thomas-d'Aquin: €31.61/m²
    2. Invalides: €31.39/m²
    3. Gros-Caillou: €31.13/m²
    4. Notre-Dame-des-Champs: €31.00/m²
    5. Ecole-Militaire: €30.90/m²
    6. Place-Vendôme: €29.49/m²
    7. Gaillon: €29.49/m²
    8. St-Germain-l'Auxerrois: €29.24/m²
    9. Champs-Elysées: €29.18/m²
   10. Arsenal: €29.11/m²

🏘️ Top 10 cheapest neighborhoods:
    1. La Chapelle: €21.10/m²
    2. Villette: €21.15/m²
    3. Saint-Fargeau: €21.23/m²
    4. Charonne: €21.32/m²
    5. Pont-de-Flandre: €21.44/m²
    6. Gare: €21.48/m²
    7. Amérique: €21.49/m²
    8. Bercy: €22.92/m²
    9. Bel-Air: €22.95/m²
   10. Père-Lachaise: €23.11/m²


### 🌊 Dataset 2: Risk Sectors Analysis

Flood risk areas in Paris (PPRI 2024).

In [13]:
# ===== Explore df_risk (Risk Sectors) =====
print("=" * 70)
print("🌊 RISK SECTORS DATASET")
print("=" * 70)

print(f"\n📊 Shape: {df_risk.shape[0]:,} rows × {df_risk.shape[1]} columns")

print(f"\n📋 Columns and types:")
for col in df_risk.columns:
    sample = df_risk[col].dropna().iloc[0] if len(df_risk[col].dropna()) > 0 else "N/A"
    sample_str = str(sample)[:50]
    print(f"   • {col} ({df_risk[col].dtype}): {sample_str}...")

print(f"\n❓ Missing values:")
missing = df_risk.isnull().sum()
for col in df_risk.columns:
    if missing[col] > 0:
        print(f"   • {col}: {missing[col]:,} ({missing[col]/len(df_risk)*100:.1f}%)")
    else:
        print(f"   ✅ {col}: no missing")

# Show sample
print(f"\n🔍 First 3 rows:")
df_risk.head(3)

🌊 RISK SECTORS DATASET

📊 Shape: 1,653 rows × 7 columns

📋 Columns and types:
   • objectid (int64): 1113...
   • zonage (str): BC...
   • n_sq_pprizone (int64): 1124...
   • st_area_shape (float64): 9954.333602538893...
   • st_perimeter_shape (float64): 403.36795183631745...
   • geo_shape (object): {'type': 'Feature', 'geometry': {'coordinates': [[...
   • geo_point_2d (object): {'lon': 2.362665091496228, 'lat': 48.8496560064786...

❓ Missing values:
   ✅ objectid: no missing
   ✅ zonage: no missing
   ✅ n_sq_pprizone: no missing
   ✅ st_area_shape: no missing
   ✅ st_perimeter_shape: no missing
   ✅ geo_shape: no missing
   ✅ geo_point_2d: no missing

🔍 First 3 rows:


,objectid,zonage,n_sq_pprizone,st_area_shape,st_perimeter_shape,geo_shape,geo_point_2d
0,1113,BC,1124,9954.333603,403.367952,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.363437084703579, 48.84986870820061], [2.363...","{'lon': 2.362665091496228, 'lat': 48.84965600647861}"
1,746,R,698,5515.285940,633.817114,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.2906180449018096, 48.85774375775233], [2.29...","{'lon': 2.2896701429302597, 'lat': 48.856826045190225}"
2,786,BC,822,1428.658670,221.367530,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.3539412867559872, 48.85511550527571], [2.35...","{'lon': 2.354299133034226, 'lat': 48.85486336886592}"


In [15]:
# ===== Analyze risk categories (safe version) =====
print("🔍 Risk Sector Categories")
print("=" * 70)

# Find simple categorical columns (skip complex types like dict/list)
for col in df_risk.columns:
    if df_risk[col].dtype == 'object':
        try:
            # Try to count unique values (will fail for dict/list)
            unique_count = df_risk[col].nunique()
            
            if 1 < unique_count < 20:  # Categorical column
                print(f"\n📊 Column: {col}")
                value_counts = df_risk[col].value_counts()
                for value, count in value_counts.items():
                    pct = (count / len(df_risk)) * 100
                    print(f"   • {value}: {count:,} ({pct:.1f}%)")
        except TypeError:
            # Skip columns with complex types (dict/list)
            print(f"\n⚠️  Skipped column '{col}' (complex type - dict/list)")
            continue

🔍 Risk Sector Categories

⚠️  Skipped column 'geo_shape' (complex type - dict/list)

⚠️  Skipped column 'geo_point_2d' (complex type - dict/list)


### 🌳 Dataset 3: Open Spaces Analysis

Green spaces to be planted (PLU Bioclimatique 2024).

In [16]:
# ===== Explore df_green (Open Spaces) =====
print("=" * 70)
print("🌳 OPEN SPACES DATASET")
print("=" * 70)

print(f"\n📊 Shape: {df_green.shape[0]:,} rows × {df_green.shape[1]} columns")

print(f"\n📋 Columns and types:")
for col in df_green.columns:
    sample = df_green[col].dropna().iloc[0] if len(df_green[col].dropna()) > 0 else "N/A"
    sample_str = str(sample)[:80]
    print(f"   • {col} ({df_green[col].dtype}): {sample_str}...")

print(f"\n❓ Missing values:")
missing = df_green.isnull().sum()
for col in df_green.columns:
    if missing[col] > 0:
        print(f"   • {col}: {missing[col]:,} ({missing[col]/len(df_green)*100:.1f}%)")

# Show sample
print(f"\n🔍 First 3 rows:")
df_green.head(3)

🌳 OPEN SPACES DATASET

📊 Shape: 7,268 rows × 10 columns

📋 Columns and types:
   • objectid (int64): 10773...
   • n_sq_elpv (int64): 9368...
   • n_sq_ca (int64): 20...
   • c_sec (str): AP...
   • n_pc (int64): 126...
   • c_asp (str): 20-AP-0126...
   • st_area_shape (float64): 43.036296070723864...
   • st_perimeter_shape (float64): 26.48199673485115...
   • geo_shape (object): {'type': 'Feature', 'geometry': {'coordinates': [[[2.391939596972359, 48.8702720...
   • geo_point_2d (object): {'lon': 2.3918881941727923, 'lat': 48.87027151851054}...

❓ Missing values:
   • c_sec: 132 (1.8%)
   • c_asp: 132 (1.8%)

🔍 First 3 rows:


,objectid,n_sq_elpv,n_sq_ca,c_sec,n_pc,c_asp,st_area_shape,st_perimeter_shape,geo_shape,geo_point_2d
0,10773,9368,20,AP,126,20-AP-0126,43.036296,26.481997,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.391939596972359, 48.87027207152296], [2.391...","{'lon': 2.3918881941727923, 'lat': 48.87027151851054}"
1,10775,6729,16,BS,66,16-BS-0066,16.717423,20.189263,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.265099705954975, 48.85270441107049], [2.265...","{'lon': 2.265055511535817, 'lat': 48.8527272782648}"
2,10786,131,17,DP,142,17-DP-0142,25.929504,22.479009,"{'type': 'Feature', 'geometry': {'coordinates': [[[2.3225564576570843, 48.896117458448614], [2.3...","{'lon': 2.322497698304243, 'lat': 48.89612169323919}"


## 📊 Step 6: Final Summary & Insights

### Key findings across all datasets:

In [17]:
# ===== Final Summary =====
print("=" * 70)
print("📊 FINAL SUMMARY — opendata.paris Integration")
print("=" * 70)

print("\n🎯 What we learned:\n")

# Rent Control insights
if df_rent is not None:
    avg_ref = df_rent['ref'].mean()
    max_ref = df_rent['ref'].max()
    min_ref = df_rent['ref'].min()
    n_quartiers = df_rent['nom_quartier'].nunique() if 'nom_quartier' in df_rent.columns else 0
    
    print("🏠 1. RENT CONTROL:")
    print(f"   • {len(df_rent):,} records covering Paris rent regulations")
    print(f"   • {n_quartiers} different neighborhoods")
    print(f"   • Reference price range: €{min_ref:.2f} - €{max_ref:.2f}/m²")
    print(f"   • Average reference: €{avg_ref:.2f}/m²")
    print()

# Risk Sectors insights
if df_risk is not None:
    print("🌊 2. RISK SECTORS (Flood Risk):")
    print(f"   • {len(df_risk):,} risk zones in Paris")
    print(f"   • {df_risk.shape[1]} attributes per zone")
    print(f"   • Source: PLU Bioclimatique 2024 (PPRI)")
    print()

# Open Spaces insights
if df_green is not None:
    print("🌳 3. OPEN SPACES (Green Spaces):")
    print(f"   • {len(df_green):,} green space records")
    print(f"   • {df_green.shape[1]} attributes per space")
    print(f"   • Source: PLU Bioclimatique 2024 (ELPV)")
    print()

print("=" * 70)
print("💡 INSIGHTS FOR THE PROJECT:")
print("=" * 70)
print("""
1. ✅ All 3 datasets are HIGH QUALITY:
   - Minimal missing values
   - Consistent structure
   - Geographic coverage of Paris

2. ✅ DATA CAN BE LINKED:
   - Rent prices ↔ Risk areas ↔ Green spaces
   - Common spatial reference (Paris districts)
   - Foundation for ETL pipeline (Step 3)

3. ✅ READY FOR STEP 2 (Database Modeling):
   - Clear entities identified
   - Relationships can be established
   - Suitable for 3NF design
""")

📊 FINAL SUMMARY — opendata.paris Integration

🎯 What we learned:

🏠 1. RENT CONTROL:
   • 10,000 records covering Paris rent regulations
   • 80 different neighborhoods
   • Reference price range: €14.30 - €41.80/m²
   • Average reference: €26.40/m²

🌊 2. RISK SECTORS (Flood Risk):
   • 1,653 risk zones in Paris
   • 7 attributes per zone
   • Source: PLU Bioclimatique 2024 (PPRI)

🌳 3. OPEN SPACES (Green Spaces):
   • 7,268 green space records
   • 10 attributes per space
   • Source: PLU Bioclimatique 2024 (ELPV)

💡 INSIGHTS FOR THE PROJECT:

1. ✅ All 3 datasets are HIGH QUALITY:
   - Minimal missing values
   - Consistent structure
   - Geographic coverage of Paris

2. ✅ DATA CAN BE LINKED:
   - Rent prices ↔ Risk areas ↔ Green spaces
   - Common spatial reference (Paris districts)
   - Foundation for ETL pipeline (Step 3)

3. ✅ READY FOR STEP 2 (Database Modeling):
   - Clear entities identified
   - Relationships can be established
   - Suitable for 3NF design

